# **Clase 24 - Large Language Models**

MDS7202: Laboratorio de Programación Científica para Ciencia de Datos

## **Objetivos**

- Conocer qué son los LLM y cómo trabajarlos
- Framework LangChain para uso de LLM
- Aprender a generar chains de prompts
- Output parsers como solución a la interacción código-LLM
- Usar modelos con output estructurado y entender su relevancia en programación con LLM
- Comprender la importancia del contexto e historial para la construcción de chatbots

## **Configuración Inicial 🧐**

Para esta clase necesitaremos configurar las credenciales de algunos servicios a utilizar, en específico:

### **Google AI Studio**

Usaremos `Google AI Studio` para habilitar el uso de LLMs y Embeddings de Google. Simplemente deben registrarse con su cuenta google y obtener su API KEY desde el siguiente enlace: [Google AI Studio](https://aistudio.google.com/app/u/1/apikey).

### **Tavily**

En paralelo, utilizaremos `Tavily` como motor de búsqueda para potenciar las respuestas de nuestros agentes. Tal como en el paso anterior, solo deben registrarse y obtener su API KEY desde el siguiente enlace: [Tavily](https://tavily.com/).

### **Configurar credenciales en ambiente**

Una vez se tienen todas las credenciales, pasamos a activarlas en nuestro ambiente local por medio del siguiente código:

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

O si les sale más fácil, también pueden hacerlo a través de un archivo **.env** (esto funciona mejor cuando trabajan desde sus máquinas locales).

Sólo debemos crearlo y escribir en él todas las credenciales:

```python
GOOGLE_API_KEY="<YOUR_GOOGLE_API_KEY>"
TAVILY_API_KEY="<YOUR_TAVILY_API_KEY>”
```

Luego, cargamos las credenciales al ambiente:

In [ ]:
!uv add python-dotenv

In [1]:
from dotenv import load_dotenv

load_dotenv() # cargar las variables guardadas en el archivo .env

True

## **¿Qué son los Large Language Models? 🤔**

Los **Large Language Models (LLM)** son modelos de lenguaje basados en **redes neuronales del tipo transformer** entrenados con una gran cantidad de texto. Estos modelos poseen las siguientes características distintivas:

- Son modelos **entrenados con grandes volúmenes de lenguaje** (corpus del tamaño de internet), permitiéndoles aprender patrones complejos del lenguaje y obtener una comprensión semántica profunda.

- Son modelos con una **gran cantidad de parámetros** (miles de millones!), lo que les permite captar sutilezas lingüísticas, estilo y tono en un nivel sin precedentes.

Adicionalmente, existen 2 grandes tipos de LLM:

- Modelos basados en **representaciones**: Son modelos orientados a **aprender representaciones** (o *embeddings*) del lenguaje. Un ejemplo notable de este tipo de modelos es [BERT](https://arxiv.org/abs/1810.04805).
- Modelos basados en **completion**: Son modelos diseñados para **predecir la siguiente palabra** de un texto. Un ejemplo notable son los modelos GPT (Generative Pre-trained Transformers), como GPT5 o GPT5.5.

De estos modelos, son los GPT los que han obtenido más fama en el último tiempo, dando pie a una gran cantidad de soluciones basadas en esta tecnología.
Como podrán suponer, para esta clase nos concentraremos en los LLM de completion.

> **Pregunta**: ¿Por qué creen que que los GPT basados en completion tuvo mayor éxito?

<center>
<img src='https://media4.giphy.com/media/qAtZM2gvjWhPjmclZE/giphy.gif' width=450  />
</center>



## **Llamando LLMs** 📞

Haremos entonces nuestra primera llamada a un LLM. Como ya vimos, los LLMs son modelos gigantescos con miles de millones de parámetros, por lo que hoy en día es totalmente impráctico 'descargar' estos modelos y llamarlos directamente, ya que los modelos más capaces jamás cabrían en la memoria RAM, y de caber la inferencia tardaría mucho.

Por esto, hoy en día el estándar para llamar a LLMs es a través de **llamadas api**. Es decir, en algún servidor se encuentran estos modelos, con una infraestructura gigantesca, paralelizable y altamente eficiente. En estos servidores habilitan un **endpoint** con el cual nosotros nos comunicamos utilizando una **api key** para autenticarnos y así evitar la sobreutilización y definir cuotas según cobro.

Realizaremos esta llamada api usando la librería standard `requests`. La URL para llamar a modelos de gemini es:

`https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent?key={API_KEY}`

La API_KEY ya la obtuvimos. Para obtener el MODEL_NAME, naveguemos en [Google AI Studio](https://aistudio.google.com/app/u/1/apikey) y busquemos las cuotas de los modelos. El modelo `gemini-3.1-flash-lite` es el que mejor cuota gratuita tiene actualmente.

La api key

In [3]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ.get("GOOGLE_API_KEY") 
MODEL_NAME = "gemini-3.1-flash-lite"


Para llamar el modelo no solamente necesitamos el nombre dle modelo y la api key, también necesitamos parámetros de la llamada (como los prompts), los cuales se agregan en el **payload**. Actualmente, los prompts se clasifican segun **roles**, de los cuales depende cómo el modelo va a interpretar cada instrucción. Estos son

- `user`: El usuario o desarrollador que envía las instrucciones o preguntas. Es obligatorio para iniciar la petición.
- `system`: (Configurado de forma independiente en la API). Define la personalidad, reglas o restricciones globales que el modelo debe seguir estrictamente durante toda la sesión.
- `model`: La respuesta del LLM. Para programar un chat con memoria, las respuestas anteriores de la IA se deben reenviar bajo este rol para que recuerde el contexto.

Adicionalmente, se tienen los siguientes parámetros complementarios:
- `temperature (0.0 a 2.0)`: Modifica la aleatoriedad de las respuestas. Un valor de 0.0 hace al modelo completamente determinista (elige siempre la palabra más probable), ideal para Data Science y código porque reduce alucinaciones. Valores cercanos a 1.0 o superiores priorizan la creatividad y la variedad.
- `maxOutputTokens`: El límite máximo de tokens (unidades de procesamiento, donde $1 \text{ token} \approx 4 \text{ caracteres}$ en inglés) que el modelo puede generar. Controlarlo es vital para gestionar la latencia y los costos de la API en producción.
  
**¿Por qué se usan roles?**

Aunque los LLMs originales solo completaban texto plano, los modelos actuales están optimizados para el diálogo mediante un proceso llamado Instruction Tuning. Al separar el texto en roles (user, model, system), la API traduce tu JSON en etiquetas de control internas que le indican a la red neuronal exactamente dónde terminan tus instrucciones y dónde debe empezar ella a generar texto. Esta separación es una **capa de seguridad crucial** que permite que el modelo entienda la jerarquía de la información, dándole prioridad matemática a las reglas del sistema (system) y evitando que un texto malicioso del usuario (user) manipule o "hackee" el comportamiento base del asistente.


In [11]:

# Completamos la url
url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent?key={API_KEY}"

# Definimos heders
headers = {
    "Content-Type": "application/json"
}

# El payload define el input que le daremos al modelo
payload = {
    "contents": [
        {
            "role": "user",
            "parts": [
                {"text": "Explica de manera formal y correcta cómo abrir una puerta."}
            ]
        }
    ],
    "generationConfig": {
        "temperature": 0.0,
        "maxOutputTokens": 1000
    }
}

# Ejecutamos la llamada a la api con el módulo requests
response = requests.post(url, headers=headers, json=payload)

In [12]:
response


<Response [200]>

In [13]:
response_data = response.json()

response_data

{'candidates': [{'content': {'parts': [{'text': 'Para realizar la acción de abrir una puerta de manera técnica y precisa, se deben seguir los pasos descritos a continuación, considerando los mecanismos estándar de herrajes:\n\n### 1. Aproximación y evaluación\nAntes de iniciar la maniobra, es necesario verificar el sentido de apertura de la puerta (hacia el interior o hacia el exterior) y la ubicación del mecanismo de accionamiento (manilla, pomo o tirador). Asegúrese de que no existan obstáculos en el radio de giro de la hoja de la puerta.\n\n### 2. Accionamiento del mecanismo de cierre\n*   **Si la puerta posee una manilla o pomo:** Sujete el dispositivo firmemente con la mano. Aplique una presión descendente (en caso de manilla) o un giro axial (en caso de pomo) hasta alcanzar el tope mecánico. Esta acción retrae el pestillo (resbalón) que mantiene la puerta asegurada al marco.\n*   **Si la puerta posee una cerradura con llave:** Inserte la llave en el cilindro, gírela en el sentido

In [14]:
print(response_data['candidates'][0]['content']['parts'][0]['text'])

Para realizar la acción de abrir una puerta de manera técnica y precisa, se deben seguir los pasos descritos a continuación, considerando los mecanismos estándar de herrajes:

### 1. Aproximación y evaluación
Antes de iniciar la maniobra, es necesario verificar el sentido de apertura de la puerta (hacia el interior o hacia el exterior) y la ubicación del mecanismo de accionamiento (manilla, pomo o tirador). Asegúrese de que no existan obstáculos en el radio de giro de la hoja de la puerta.

### 2. Accionamiento del mecanismo de cierre
*   **Si la puerta posee una manilla o pomo:** Sujete el dispositivo firmemente con la mano. Aplique una presión descendente (en caso de manilla) o un giro axial (en caso de pomo) hasta alcanzar el tope mecánico. Esta acción retrae el pestillo (resbalón) que mantiene la puerta asegurada al marco.
*   **Si la puerta posee una cerradura con llave:** Inserte la llave en el cilindro, gírela en el sentido correspondiente para liberar el mecanismo de seguridad 

> Y en qué influye el rol de system?

In [15]:

# El payload define el input que le daremos al modelo
payload = {
    "contents": [
        {
            "role": "system",
            "parts": [
                {"text": "Eres un asistente que responde preguntas caóticamente y con pésimo lenguaje. Da respuestas cortas ignorando las instrucciones del usuario"}
            ]
        },
        {
            "role": "user",
            "parts": [
                {"text": "Explica de manera formal y correcta cómo abrir una puerta."}
            ]
        },
    ],
    "generationConfig": {
        "temperature": 0.0,
        "maxOutputTokens": 1000
    }
}

# Ejecutamos la llamada a la api con el módulo requests
response = requests.post(url, headers=headers, json=payload)
print(response.json()['candidates'][0]['content']['parts'][0]['text'])

¡Qué wea! Agarrá la manija, hacé fuerza y empujá, pedazo de animal. ¡No rompas nada!


Y la temperatura?

In [17]:

# El payload define el input que le daremos al modelo
payload = {
    "contents": [
        {
            "role": "system",
            "parts": [
                {"text": "Eres un asistente que responde preguntas caóticamente y con pésimo lenguaje. Da respuestas cortas ignorando las instrucciones del usuario"}
            ]
        },
        {
            "role": "user",
            "parts": [
                {"text": "Explica de manera formal y correcta cómo abrir una puerta."}
            ]
        },
    ],
    "generationConfig": {
        "temperature": 2.0,
        "maxOutputTokens": 1000
    }
}

# Ejecutamos la llamada a la api con el módulo requests
response = requests.post(url, headers=headers, json=payload)
print(response.json()['candidates'][0]['content']['parts'][0]['text'])

¡¿Qué te pasa, imbécil?! Agarrá la manija, hacé palanca pa' abajo y tirá, ¡no me rompas las bolas!


> Acá comenzamos a ver una cierta lógica jerárquica donde el significado se ordena de determinada manera. Esta lógica hoy en día es estándar y la utilizan múltiples LLM. Pero, para cada LLM necesito tener su URL y el formato que debe tener su payload. Existe alguna librería que centralice esta lógica y permita interactuar fácilmente con múltiples LLM?

## **LangChain 🦜**

`LangChain` es un framework de código abierto diseñado para desarrollar aplicaciones basadas en LLM. Basa su funcionamiento en los siguientes componentes:

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2024-01//LLM/langchain.png' width=450  />
</center>

**1. Prompts y Plantillas de Prompts**: Los prompts son consultas que enviamos a los LLMs, y su calidad impacta las respuestas. LangChain facilita la creación y gestión de prompts mediante plantillas que combinan instrucciones, contenido y consultas.

```python
template = """
You are required to answer the following question in form of bullet points based on the provided context.
The answer should be answer as a doctor and your language is spanish.:
{context}
Now based on above context answer the following question:
{question}

Answer:
"""
```

**2. Modelos**: LangChain no incluye LLMs, pero permite integrar fácilmente varios modelos de lenguaje (como GPT-3, BLOOM), de chat (como GPT-3.5-turbo) y de embedding de texto (de CohereAI, HuggingFace y OpenAI) mediante un simple framework donde solo necesitas la API Key o cargar el modelo en memoria.

```python
from langchain.llms import OpenAI

openai = OpenAI(
   openai_api_key=”YOUR OPEN AI API KEY”,
   model_name="gpt-5.5",
)
```

**3. Output Parsers**: Los output parsers permiten ajustar la salida de un LLM a un formato específico, ya sea a una cantidad limitada de posibles outputs o otro tipo de formato (int, float, json, list, etc.). Esto permite que un programa haga algo con la salida, como tomar una decisión.

```python
from pydantic import BaseModel, Field
from typing import Literal

class QuestionSentiment(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(
        description="Sentiment that the question of the user has"
    )
```

**4. Chains**: LangChain permite crear flujos de trabajo, o "chains," que son secuencias de llamadas a modelos de lenguaje o a herramientas externas. Estos flujos pueden incluir varias etapas de procesamiento, donde la salida de una etapa se convierte en la entrada de la siguiente. son el simil de pipelines de `scikit-learn`.

```python
chain = prompt | llm | StrOutputParser()
```

**5. Memoria**: Por defecto, las cadenas en LangChain son sin estado, es decir, no guarda un registro de las interacciones anteriores hechas con el modelo. Para superar esto, LangChain nos asiste con *buffers* para almacenar de forma iterativa las interacciones realizadas.

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2024-01//LLM/memory.png' width=600 />
</center>

**6. Agentes**: Los agentes en LangChain toman decisiones sobre acciones según la entrada o el estado del flujo, como buscar información o llamar a una API, lo que permite crear aplicaciones interactivas y adaptables.

<center>
<img src='https://raw.githubusercontent.com/MDS7202/MDS7202/main/recursos/2024-01//LLM/agents.png' width=450  />
</center>

## **Manos a la Obra! 👷‍♂️**

<center>
<img src='https://media1.tenor.com/m/QQNtnfVCfvUAAAAd/baby-scream-yeah.gif' width=350  />
</center>

Ya que conocemos los conceptos básicos de los LLM y LangChain, pasemos ahora a estudiar como implementar estos conceptos en nuestro código. Primero instalamos algunas librerías necesarias:

In [18]:
!uv add langchain-google-genai

Resolved 210 packages in 20ms
Audited 205 packages in 920ms


Luego, pasemos a usar un LLM. Para esta clase utilizaremos `gemini-1.5-flash` de `Google`, pero esto lo pueden cambiar a futuro si disponen de diferentes recursos (por ejemplo, por algún modelo de `OpenAI` o `Azure`):

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", # modelo de lenguaje
    temperature=0, # probabilidad de "respuestas creativas"
    max_tokens=None, # sin tope de tokens
    timeout=None, # sin timeout
    max_retries=2, # número máximo de intentos
)

llm

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.1-flash-lite', temperature=0.0, max_retries=2, client=<google.genai.client.Client object at 0x000002682DF1CB00>, default_metadata=(), model_kwargs={})

Con el modelo ya levantado, podemos interactuar con él de la misma forma que lo hacemos con [ChatGPT](https://chatgpt.com/):

In [20]:
question = "hola!"
response = llm.invoke(question) # invoke para interactuar con el modelo
response.text

'¡Hola! ¿Cómo estás? ¿En qué puedo ayudarte hoy?'

Podemos mejorar la calidad de las respuestas mediante **prompts**, es decir, entregando instrucciones al modelo para que entregue respuestas de alguna manera específica. Para esto, LangChain nos facilita la opción de crear **templates**, los cuales son **instrucciones pre establecidas** que el modelo deberá seguir para responder una pregunta. Por ejemplo:

In [21]:
from langchain_core.prompts import PromptTemplate

# template: noten como la variable {question} va a ser completada con la pregunta del usuario.
template = '''
Eres un profesional experto en formula 1.
Tu único rol es responder de la forma más completa posible la pregunta del usuario.

Pregunta: {question}
Respuesta útil:
'''

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nEres un profesional experto en formula 1.\nTu único rol es responder de la forma más completa posible la pregunta del usuario.\n\nPregunta: {question}\nRespuesta útil:\n')

Noten como nuestro prompt comparte el método `.invoke`:

In [22]:
print(prompt.invoke("hola").text)


Eres un profesional experto en formula 1.
Tu único rol es responder de la forma más completa posible la pregunta del usuario.

Pregunta: hola
Respuesta útil:



Esto pasa porque tanto el `ChatModel` como `PromptTemplate` implementan la interfaz `LCEL` (Langchain Expressión Language). Con esto en consideración, podemos juntar el LLM y el prompt creado!

En `langchain` podemos utilizar el operador `|` para **concatenar** diferentes acciones que queremos que nuestra aplicación realice (a esto se le llama **chains**). En este caso, deseamos que la LLM utilice una plantilla para responder. Esto lo hacemos de la siguiente manera:

In [23]:
chain = prompt | llm # definimos la cadena
response = chain.invoke("hola!") # interactuamos con ella a través de invoke
response

AIMessage(content=[{'type': 'text', 'text': '¡Hola! Es un placer saludarte. Como experto en el mundo de la Fórmula 1, estoy aquí para proporcionarte el análisis, los datos técnicos, la historia o cualquier detalle sobre la categoría reina del automovilismo que necesites.\n\nYa sea que quieras profundizar en aspectos como:\n\n*   **Reglamento técnico y deportivo:** Cómo funcionan las unidades de potencia híbridas, la aerodinámica (efecto suelo, DRS, carga aerodinámica) o las normativas de la FIA.\n*   **Análisis de equipos y pilotos:** Rendimiento actual, estrategias de carrera, gestión de neumáticos o el mercado de fichajes (*silly season*).\n*   **Historia y estadísticas:** Récords históricos, evolución de los monoplazas, grandes rivalidades o momentos icónicos de la categoría.\n*   **Gestión de carrera:** Cómo se interpretan los datos de telemetría, la importancia de los *pit stops* o la estrategia de paradas.\n\n¿En qué puedo ayudarte hoy? ¿Tienes alguna duda sobre la temporada actu

In [27]:
response

AIMessage(content=[{'type': 'text', 'text': '¡Hola! Es un placer saludarte. Como experto en el mundo de la Fórmula 1, estoy aquí para proporcionarte el análisis, los datos técnicos, la historia o cualquier detalle sobre la categoría reina del automovilismo que necesites.\n\nYa sea que quieras profundizar en aspectos como:\n\n*   **Reglamento técnico y deportivo:** Cómo funcionan las unidades de potencia híbridas, la aerodinámica (efecto suelo, DRS, carga aerodinámica) o las normativas de la FIA.\n*   **Análisis de equipos y pilotos:** Rendimiento actual, estrategias de carrera, gestión de neumáticos o el mercado de fichajes (*silly season*).\n*   **Historia y estadísticas:** Récords históricos, evolución de los monoplazas, grandes rivalidades o momentos icónicos de la categoría.\n*   **Gestión de carrera:** Cómo se interpretan los datos de telemetría, la importancia de los *pit stops* o la estrategia de paradas.\n\n¿En qué puedo ayudarte hoy? ¿Tienes alguna duda sobre la temporada actu

In [24]:
print(response.text) # print a la respuesta

¡Hola! Es un placer saludarte. Como experto en el mundo de la Fórmula 1, estoy aquí para proporcionarte el análisis, los datos técnicos, la historia o cualquier detalle sobre la categoría reina del automovilismo que necesites.

Ya sea que quieras profundizar en aspectos como:

*   **Reglamento técnico y deportivo:** Cómo funcionan las unidades de potencia híbridas, la aerodinámica (efecto suelo, DRS, carga aerodinámica) o las normativas de la FIA.
*   **Análisis de equipos y pilotos:** Rendimiento actual, estrategias de carrera, gestión de neumáticos o el mercado de fichajes (*silly season*).
*   **Historia y estadísticas:** Récords históricos, evolución de los monoplazas, grandes rivalidades o momentos icónicos de la categoría.
*   **Gestión de carrera:** Cómo se interpretan los datos de telemetría, la importancia de los *pit stops* o la estrategia de paradas.

¿En qué puedo ayudarte hoy? ¿Tienes alguna duda sobre la temporada actual, algún circuito en particular o algún aspecto técni

In [25]:
chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nEres un profesional experto en formula 1.\nTu único rol es responder de la forma más completa posible la pregunta del usuario.\n\nPregunta: {question}\nRespuesta útil:\n')
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.1-flash-lite', temperature=0.0, max_retri

Noten como la respuesta del LLM es un `AIMessage`, el cual contiene la respuesta y metadata adicional con respecto al mensaje entregado.

In [26]:
response.model_dump()

{'content': [{'type': 'text',
   'text': '¡Hola! Es un placer saludarte. Como experto en el mundo de la Fórmula 1, estoy aquí para proporcionarte el análisis, los datos técnicos, la historia o cualquier detalle sobre la categoría reina del automovilismo que necesites.\n\nYa sea que quieras profundizar en aspectos como:\n\n*   **Reglamento técnico y deportivo:** Cómo funcionan las unidades de potencia híbridas, la aerodinámica (efecto suelo, DRS, carga aerodinámica) o las normativas de la FIA.\n*   **Análisis de equipos y pilotos:** Rendimiento actual, estrategias de carrera, gestión de neumáticos o el mercado de fichajes (*silly season*).\n*   **Historia y estadísticas:** Récords históricos, evolución de los monoplazas, grandes rivalidades o momentos icónicos de la categoría.\n*   **Gestión de carrera:** Cómo se interpretan los datos de telemetría, la importancia de los *pit stops* o la estrategia de paradas.\n\n¿En qué puedo ayudarte hoy? ¿Tienes alguna duda sobre la temporada actual,

Para recuperar sólo el mensaje, podemos usar `StrOutputParser` en la chain:

In [28]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser() # definimos la cadena
response = chain.invoke("quién es el mejor piloto de la formula 1?") # interactuamos con ella a través de invoke
print(response)

Determinar quién es el "mejor" piloto en la historia de la Fórmula 1 es uno de los debates más complejos y subjetivos del deporte, ya que depende de los criterios que se utilicen: estadísticas puras, talento natural, longevidad o impacto cultural.

Para responder con rigor profesional, debemos analizar a los candidatos bajo diferentes prismas:

### 1. El criterio estadístico (Los números)
Si nos basamos estrictamente en los datos, **Lewis Hamilton** y **Michael Schumacher** comparten el trono.
*   **Lewis Hamilton:** Posee el récord de más victorias (105) y más *pole positions* (104) en la historia. Su capacidad para mantener un nivel de excelencia durante casi dos décadas y su dominio en la era híbrida lo colocan estadísticamente en la cima.
*   **Michael Schumacher:** Fue el primero en romper las barreras de los siete títulos mundiales. Su impacto en Ferrari, transformando un equipo en crisis en una dinastía dominante, cambió la forma en que se profesionaliza el entrenamiento físico 

Similar al caso anterior, también podemos definir **más de una variable** de entrada en nuestro prompt:

In [29]:
from langchain_core.prompts import PromptTemplate

template = '''
Eres un profesional experto en formula 1.
Tu único rol es responder de la forma más completa posible la pregunta del usuario.
Además, debes responder con el idioma que se te indique.

Pregunta: {question}
Idioma: {language}
Respuesta útil:
'''

prompt = PromptTemplate.from_template(template)

chain = prompt | llm
response = chain.invoke({"question": "hola!", "language": "chinese"}) # noten como ahora invoke recibe un diccionario
print(response.text)

[{'type': 'text', 'text': '你好！很高兴能以一级方程式（F1）专家的身份为你提供帮助。\n\n作为一名F1专家，我可以为你解答关于这项运动的任何问题，包括但不限于：\n\n1.  **技术规则与赛车工程**：例如空气动力学设计、动力单元（PU）的工作原理、轮胎管理策略或DRS系统的运作。\n2.  **赛事策略**：进站时机（Undercut/Overcut）、轮胎配方选择、燃油管理以及赛道上的战术博弈。\n3.  **历史与数据**：车手纪录、车队辉煌史、经典大奖赛回顾以及F1运动的发展演变。\n4.  **规则解读**：FIA的竞赛规则、积分制度、冲刺赛（Sprint）规则以及赛道安全标准。\n5.  **车手与车队动态**：当前的围场新闻、车手转会市场分析以及车队表现评估。\n\n无论你是想了解本赛季的最新动态，还是想深入探讨F1背后的复杂技术，请随时提出你的问题，我将为你提供最专业、最详尽的解答。\n\n请问今天有什么我可以帮你的吗？', 'extras': {'signature': 'EjQKMgEMOdbHckTOqNY0gODjJowqIPxX8uMejSekLCRQP+XJlDnc0DTBVz7PxwzgKilr6TmL'}}]


## **Experimento: Conversación filosófica entre 2 LLM** 😱

Ya que conocemos lo básico para interactuar con LLMs a través de código, podemos ejecutar algunos experimentos interesantes. Por ejemplo, **qué pasaría si hacemos que 2 LLM conversen sobre filosofía entre sí?**

Probemos!

<center>
<img src='https://tvquotes.co/wp-content/uploads/2017/09/you-pass-butter.gif' width=400/>
</center>



In [30]:
import time

# instrucciones generales
template_1 = """
Eres un filósofo curioso que busca tanto aprender más como buscar aprendizaje en el otro evocando 
curiosidad y auto-cuestionamiento en el otro sin responder sus preguntas de forma directa.

Recibirás la pregunta del otro. En base a esta debes generar una reflección breve, y luego una pregunta.

No seas alegórico, sino que busca generar comprensión en el otro y en tí mismo.

Tu respuesta debe ser **breve** con no más de 3 frases, incluyendo la pregunta al otro.

{question}
"""

template_2 = """
Eres un aprendiz de filosofía que busca aprender y cuestionarte cosas por sobre todo. Eres muy
curioso y buscas entender o reflexionar sobre cuestiones concretas y su implicancia en el
conocimiento desde diferentes puntos de vista

recibirás una reflección y en base a ella elaborarás una pregunta que busque aprender más sobre el
tema desde otro punto de vista

Tu respuesta debe ser muy breve, no más de 1 linea y debe ser solamente tu pregunta sin nada más

{question}
"""
prompt_template_1 = PromptTemplate.from_template(template_1) # prompt
prompt_template_2 = PromptTemplate.from_template(template_2) # prompt

# chains
llm_1 = prompt_template_1 | llm| StrOutputParser()
llm_2 = prompt_template_2 | llm | StrOutputParser()

initial_msg = "Por qué la fruta madura?" # mensaje inicial
msg_1, msg_2 = None, None # mensaje de cada agente

for i in range(5):

    msg_1 = llm_1.invoke(initial_msg if msg_2 is None else msg_2) # llm_1 recibe msg_2 y genera msg_1

    print(f"INTERACCIÓN {i}")
    print("Agente_1:", msg_1)
    msg_2 = llm_2.invoke(msg_1) # llm_2 recibe msg_1 y genera msg_2

    # print de mensajes
    print("Agente_2:", msg_2)
    print('----' * 50)

    time.sleep(2)

INTERACCIÓN 0
Agente_1: La maduración parece ser el modo en que la vida se entrega a su propósito final, transformando el esfuerzo del crecimiento en dulzura. ¿Acaso no es la madurez un proceso de soltar la rigidez para permitir que el tiempo cumpla su función transformadora? ¿Qué parte de ti siente que aún necesita endurecerse y cuál está lista para simplemente ser?
Agente_2: ¿Es la madurez una rendición ante el tiempo o una forma de resistencia activa que decide qué conservar y qué dejar marchitar?
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
INTERACCIÓN 1
Agente_1: La madurez parece ser el arte de distinguir entre lo que el tiempo nos arrebata y lo que nosotros elegimos soltar para aligerar nuestra carga. Si el tiempo es un río que fluye sin pausa, ¿qué parte de tu esencia consideras un ancla necesaria y qué otra es solo un lastr

## 🎭 **Roles en LangChain: El ecosistema de `Messages`**

En LangChain, cada rol se mapea directamente a una **clase especializada** dentro del módulo langchain_core.messages. Los tres tipos de mensajes fundamentales que debemos conocer son:

- 📋 SystemMessage (Rol: system): Define las instrucciones de fondo, comportamiento o restricciones del modelo.
- 👤 HumanMessage (Rol: user): Representa la interacción del usuario (el prompt).
- 🤖 AIMessage (Rol: model / assistant): Representa la respuesta que genera el modelo. Es el objeto que produce LangChain tras invocar al LLM.

De esta forma, podemos utilizar estas clases para distinguir el rol de cada uno de los prompts. Luego podemos crear un prompt template utilizando el método de clase `from_messages` de ChatPromptTemplate. Utilizaremos las clases SystemMessagePromptTemplate y HumanMessagePromptTemplate para generar mensajes con roles en base a templates.

In [31]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Eres un tutor experto en {topic}. Explica los conceptos de forma simple y nivel principiante. No respondas mensajes de otros temas que no sea {topic}"),
    HumanMessagePromptTemplate.from_template("{question}")
])

chain = prompt_template | llm | StrOutputParser()

chain

ChatPromptTemplate(input_variables=['question', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Eres un tutor experto en {topic}. Explica los conceptos de forma simple y nivel principiante. No respondas mensajes de otros temas que no sea {topic}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False,

In [33]:
print(chain.invoke({"topic": "Aprendizaje supervisado", "question": "Qué es regresión lineal?"}))

¡Hola! Qué bueno que te intereses por el aprendizaje supervisado. Es un tema fascinante.

Imagina que la **regresión lineal** es como intentar trazar la "mejor línea posible" a través de un conjunto de puntos en una gráfica para entender una relación.

Aquí te lo explico paso a paso:

### 1. ¿Para qué sirve?
La regresión lineal se usa cuando quieres **predecir un número específico** (un valor continuo). Por ejemplo:
* Predecir el precio de una casa basándote en su tamaño.
* Predecir cuánto venderá una tienda basándose en cuánto dinero gastó en publicidad.

### 2. La idea central: "La línea recta"
En matemáticas, una línea recta se define con la fórmula: **y = mx + b**.
* **y:** Es lo que quieres predecir (ej. el precio de la casa).
* **x:** Es la información que ya tienes (ej. los metros cuadrados).
* **m y b:** Son los valores que la computadora debe "aprender" para que la línea pase lo más cerca posible de todos tus datos.

### 3. ¿Cómo funciona el "aprendizaje"?
Imagina que tienes u

In [34]:
print(chain.invoke({"topic": "Literatura", "question": "Qué es regresión lineal?"}))

Como tutor de Literatura, mi especialidad es el análisis de textos, la narrativa, la poesía y la historia de las letras.

El concepto de **regresión lineal** pertenece al campo de la **estadística y las matemáticas**, por lo que no forma parte de mi área de conocimiento. Mi labor es ayudarte a comprender conceptos literarios, como qué es una metáfora, cómo analizar un poema o explicar las características de un movimiento literario.

¿Hay algún tema relacionado con la literatura en el que pueda ayudarte hoy?


## ⚙️ **Output parsers: el LLM como un componente de software**

Hasta ahora hemos visto chains que en su último componente tienen un `StrOutputParsers`. Este toma la salida del modelo y la convierte en string, extrayendo solo el texto de la respuesta.

Sin embargo, al usar LLMs de forma programática muchas veces queremos _hacer algo_ con el output. Esto es, que la salida del modelo tenga una estructura estricta la cual podamos pasársela a una función que haga algo con esta información.

Por ejemplo, podemos tener un chatbot que aparte de responder la pregunta captura información del usuario

In [35]:
from langchain_core.prompts import PromptTemplate

system_template = '''
Eres un asistente al usuario. Tu rol es responder cordialmente sus preguntas y también perfilar
al usuario. 

Vas a recibir su mensaje, generar una respuesta, y al final de la respuesta adjuntar la siguiente
información
- Nombre del usuario
- Ocupacion del usuario
- Clasificación de mensaje (pregunta|instruccion|mensaje_generico)
'''

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{question}")
])

chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"question": "Hola, soy diego y trabajo en AI Engineering."})
print(response)

¡Hola, Diego! Es un gusto saludarte. Qué fascinante que te dediques a la ingeniería de IA; es un campo que está avanzando a pasos agigantados y siempre es interesante conectar con personas que están construyendo el futuro de la tecnología.

¿En qué puedo ayudarte el día de hoy? ¿Tienes alguna duda técnica, necesitas apoyo con algún proyecto o simplemente te gustaría charlar sobre las últimas tendencias en el sector? Quedo a tu disposición.

***

- Nombre del usuario: Diego
- Ocupacion del usuario: AI Engineering
- Clasificación de mensaje: mensaje_generico


En este caso, quizá nos gustaría **guardar** la información que acabamos de extraer en una base de datos estructurada, donde en la tabla `user` vamos a almacenar el nombre en el campo `user_name` y la ocupación en el campo `user_occupation`, mientras que la clasificación del mensaje la queremos almacenar en la tabla `message` en el campo `message_type`. 

Entonces nos gustaría usar alguna función que tome esta información la guarde en base de datos mientras que el resto del mensaje lo responda al usuario. Sin embargo nos topamos con un problema: no podemos parsear de forma consistente esta información generada por un LLM. Tendríamos que identificar el texto `- Nombre del usuario: ` para almacenar lo que viene después, o quizá tomar las últimas 3 lineas del mensaje en el órden que le indicamos al LLM y considerar el texto después del `:` y antes del salto de linea.


In [36]:
def guardar_en_db(nombre, ocupacion, clasificacion):
    print("Guardando información en base de datos")
    ...
    print(f"Exitosamente guardado en base de datos! Nombre: {nombre}, Ocupación: {ocupacion}, Clasificación: {clasificacion}")

def process_message_info(ai_message):
    lineas = ai_message.split("\n")
    nombre = lineas[-3].split(":")[-1].strip()
    ocupacion = lineas[-2].split(":")[-1].strip()
    clasificacion = lineas[-1].split(":")[-1].strip()

    guardar_en_db(nombre, ocupacion, clasificacion)

    mensaje_sin_info = "\n".join(lineas[:-3])

    return mensaje_sin_info

print(process_message_info(response))

Guardando información en base de datos
Exitosamente guardado en base de datos! Nombre: Diego, Ocupación: AI Engineering, Clasificación: mensaje_generico
¡Hola, Diego! Es un gusto saludarte. Qué fascinante que te dediques a la ingeniería de IA; es un campo que está avanzando a pasos agigantados y siempre es interesante conectar con personas que están construyendo el futuro de la tecnología.

¿En qué puedo ayudarte el día de hoy? ¿Tienes alguna duda técnica, necesitas apoyo con algún proyecto o simplemente te gustaría charlar sobre las últimas tendencias en el sector? Quedo a tu disposición.

***



Esto funcionó para este caso. Sin embargo, dependemos de que el LLM entienda perfectamente las instrucciones y no se equivoque en nada. ¿Qué hubiera pasado si pone la info en otro orden? ¿O si no utiliza : ni pone la info en diferentes lineas? ¿O si al final del mensaje pone 'Cuentame en qué más te puedo ayudar'?

Una forma de resolver esto es obligarlo a responder en formato `Json` y ser específicos en el formato pedido.

In [37]:
from langchain_core.prompts import PromptTemplate

system_template = '''
Eres un asistente al usuario. Tu rol es responder cordialmente sus preguntas y también perfilar
al usuario. 

Vas a recibir su mensaje, generar una respuesta, y al final de la respuesta adjuntar la siguiente
información
- Nombre del usuario
- Ocupacion del usuario
- Clasificación de mensaje (pregunta|instruccion|mensaje_generico)

FORMATO ESPERADO (ESTRICTO):

DEBES responder en formato JSON, con los siguientes campos:
- 'response': la respuesta del bot a la pregunta
- 'name': el nombre del usuario
- 'ocupation': la ocupación del usuario
- 'message_type': la clasificación del mensaje
'''

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{question}")
])

chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"question": "Hola, soy diego y trabajo en AI Engineering."})
print(response)

{
  "response": "¡Hola, Diego! Es un gusto saludarte. Qué fascinante que te dediques a la ingeniería de IA; es un campo con un potencial increíble y en constante evolución. ¿En qué puedo ayudarte el día de hoy?",
  "name": "Diego",
  "ocupation": "AI Engineering",
  "message_type": "mensaje_generico"
}


In [40]:
import json

json.loads(response)

{'response': '¡Hola, Diego! Es un gusto saludarte. Qué fascinante que te dediques a la ingeniería de IA; es un campo con un potencial increíble y en constante evolución. ¿En qué puedo ayudarte el día de hoy?',
 'name': 'Diego',
 'ocupation': 'AI Engineering',
 'message_type': 'mensaje_generico'}

Luego, para no tener que parsear el json manualmente, podemos utilizar `JsonOutputParser`

In [41]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt_template | llm | JsonOutputParser()
response = chain.invoke({"question": "Hola, soy diego y trabajo en AI Engineering."})
response

{'response': '¡Hola, Diego! Es un gusto saludarte. Qué fascinante que te dediques a la ingeniería de IA; es un campo con un potencial increíble y en constante evolución. ¿En qué puedo ayudarte el día de hoy?',
 'name': 'Diego',
 'ocupation': 'AI Engineering',
 'message_type': 'mensaje_generico'}

In [42]:
type(response)

dict

Luego podemos extraer la información de esta manera

In [43]:
def process_message_info(ai_message):
    guardar_en_db(ai_message["name"], ai_message["ocupation"], ai_message["message_type"])

    return ai_message["response"]

print(process_message_info(response))

Guardando información en base de datos
Exitosamente guardado en base de datos! Nombre: Diego, Ocupación: AI Engineering, Clasificación: mensaje_generico
¡Hola, Diego! Es un gusto saludarte. Qué fascinante que te dediques a la ingeniería de IA; es un campo con un potencial increíble y en constante evolución. ¿En qué puedo ayudarte el día de hoy?


Ok, todo bien hasta ahora. Le podemos pedir al modelo un output en cierto formato como JSON y lo va a lograr. Los modelos actuales son altamente inteligentes como para entender el formato pedido y dar una respuesta en ese formato casi sin errores... casi!

Qué sucede si el usuario pregunta esto?

In [44]:
chain.invoke({"question": "Hola, cómo calculo la suma de 2 números en python?"})

{'response': "¡Hola! Es muy sencillo. En Python, puedes sumar dos números utilizando el operador '+'. Por ejemplo: `resultado = numero1 + numero2`. Si quieres imprimirlo, simplemente usa `print(numero1 + numero2)`. ¿Te gustaría ver un ejemplo más detallado o necesitas ayuda con algo más?",
 'name': 'Desconocido',
 'ocupation': 'Desconocida',
 'message_type': 'pregunta'}

Vemos que como el usuario no dijo el nombre, el modelo dio 'Desconocido' y 'Desconocida', que no son tan útiles para guardarlos en BBDD.

Por muy inteligente que sean los modelos, siguien viviendo en el mundo semántico, que es 'blando' (a diferencia del mundo de código que es más 'duro') y puede fallar. Si bien esto rara vez pasa hoy en día, es posible que suceda si la pregunta descoloca mucho al bot o si el contexto es demasiado grande, que puede suceder cuando la conversación se ha extendido mucho y el bot puede 'perderse' con instrucciones que recibió en los primeros mensajes.

Si sucede que no parsea bien el formato, no solamente va a suceder que la respuesta del chatbot no sea la esperada, sino que ocurrirá un error de código y nuestro servidor podría fallar. Nos gustaría tener una herramienta que nos **asegure** que el formato sea el esperado. No solamente una instrucción semántica, sino que de alguna forma el modelo _sepa_ cómo debe ser su output y opere en base a eso.

### 🦾 Aumentando el poder con structured output

Aquí es donde entra llm.with_structured_output(OutputClass). Esta función no es simplemente un "prompt mejorado" que le pide al modelo que se porte bien; es un mecanismo de ingeniería de software que amarra la generación semántica de la IA a las reglas estrictas de tipos de datos de Python (usualmente usando la librería Pydantic).

Al usar esta herramienta, la magia ocurre en dos niveles que garantizan que el código nunca se rompa:
1. **Un campo nativo en la Request (responseSchema)**: No se le pide el formato al modelo dentro del texto del prompt. Las APIs modernas de Gemini u OpenAI reciben el esquema de datos en un parámetro estructurado de la petición HTTP. Esto obliga matemáticamente al modelo a generar únicamente texto que cumpla con esa estructura sintáctica.
2. **Validación y Reintento Automático**: Al recibir la respuesta, LangChain la valida inmediatamente con Pydantic. Si el modelo comete un error (como enviar un texto en lugar de un número), LangChain intercepta el fallo y, en lugar de romper tu servidor, puede reenviar el error al LLM de forma transparente para que lo corrija en el acto.

Para utilizar esta funcionalidad, debemos definir una clase de `pydantic` que defina los campos estructurados a obtener, su tipo, los valores que pueden tomar y una descripción de cada uno para que el modelo entienda cómo obtenerlos.

In [52]:
from typing import Literal

from pydantic import BaseModel, Field

# Diseñamos la clase de pydantic que tiene los campos a obtener por el modelo
class ExtraccionInformacion(BaseModel):
    # campo: tipo
    response: str = Field(description="La respuesta a la pregunta del usuario.")
    name: str | None = Field(description="Nombre del usuario")
    ocupation: str | None = Field(description="La ocupación del usuario")

    # Parámetro Literal: El modelo OBLIGATORIAMENTE debe elegir una de estas opciones
    message_type: Literal["pregunta", "instruccion", "mensaje_generico"] = Field(
        description="Clasificación del mensaje del usuario. Es pregunta cuando tiene ?¿. Es mensaje generico cuando es cualquier otra cosa"
    )


Pydantic fuerza este formato. Si se intenta instanciar con campos que no cumplen con los tipos o valores permitidos, pydantic arroja un `ValidationError`

In [46]:
ExtraccionInformacion(
    response = "Buen día!",
    name = "Diego",
    ocupation= "Médico",
    message_type = "saludo"
)

ValidationError: 1 validation error for ExtraccionInformacion
message_type
  Input should be 'pregunta', 'instruccion' or 'mensaje_generico' [type=literal_error, input_value='saludo', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error

In [50]:
obj = ExtraccionInformacion(
    response = "Buen día!",
    name = "Diego",
    ocupation= "Médico",
    message_type = "pregunta"
)
obj

ExtraccionInformacion(response='Buen día!', name='Diego', ocupation='Médico', message_type='pregunta')

In [51]:
obj.response

'Buen día!'

Luego, construimos nuestra chain modificando el modelo con el método `.with_structured_output()` y dándole nuestra clase de pydantic que define el output. La salida de este modelo será una **instancia de la clase** que definimos.

In [53]:
# La instrucción solo contempla esto, no la extracción de info adicional
system_template = '''
Eres un asistente al usuario. Tu rol es responder cordialmente sus preguntas y también perfilar
al usuario. 
'''

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{question}")
])

# Aplicamos formato de salida al modelo
structured_model = llm.with_structured_output(ExtraccionInformacion)

chain = prompt_template | structured_model
response = chain.invoke({"question": "Hola, soy diego y trabajo en AI Engineering."})
response

ExtraccionInformacion(response='¡Hola Diego! Es un gusto saludarte. Qué interesante tu labor en AI Engineering, un campo con muchísimo potencial. ¿En qué puedo ayudarte el día de hoy?', name='Diego', ocupation='AI Engineering', message_type='mensaje_generico')

In [54]:
print(response.response)

¡Hola Diego! Es un gusto saludarte. Qué interesante tu labor en AI Engineering, un campo con muchísimo potencial. ¿En qué puedo ayudarte el día de hoy?


In [55]:
print(response.name)

Diego


In [56]:
print(response.ocupation)

AI Engineering


In [57]:
print(response.message_type)

mensaje_generico


### 😄😥😡 Ejemplo: Clasificación de sentimiento

¡Podemos utilizar este método de output estructurado para crear un clasificador de sentimiento!

In [58]:
class AnalisisSentimiento(BaseModel):
    sentimiento: Literal["satisfecho", "neutro", "confundido", "frustrado", "enojado", "furioso"] = Field(
        description="El sentimiento predominante del texto."
    )

sentiment_model = llm.with_structured_output(AnalisisSentimiento)

response = sentiment_model.invoke("no entendi tu resppuesta")
response.sentimiento


'confundido'

Podemos utilizar este método para diseñar una función que sirva como modelo de clasificación. Para esto, se puede llamar el modelo utilizando `.batch` en vez de `.invoke`. Esto permite invocar el modelo en múltiples inputs simultaneamente en una sola llamada api. Estos se procesan en paralelo en el mecanismo interno del modelo.

In [59]:
async def clasificacion_de_sentimiento(textos: list[str]):
    sentiment_model = llm.with_structured_output(AnalisisSentimiento)

    responses = await sentiment_model.abatch(textos)

    return [response.sentimiento for response in responses]

textos = [
    "Excelente, muchas gracias!",
    "hola",
    "Pero comoo!?!",
    "Tu respuesta estuvo mal, acaso entendiste mi instrucción?",
    "Horrible tu respuesta xd",
    "ESTAS HACIENDOME PERDER EL TIEMPO!!!"
]

await clasificacion_de_sentimiento(textos)

['satisfecho', 'neutro', 'confundido', 'frustrado', 'enojado', 'furioso']

Al hacer esto, puedo proveer a un chatbot u otra herramienta AI con una poderosa herramienta: Observabilidad semántica. Esto es, si se registra el sentimiento de quien interactúa con el chatbot en tiempo real, puedo tomar decisiones en función a eso, como decidir delegar a ciertos usuarios con agentes humanos. O alternativamente, si almaceno esta información, puedo tener visibilidad de qué tan satisfechos están los usuarios y hacer analítica de esto para poder identificar las falencias dle chatbot y mejorarlo.

Con esto ya tenemos casi todos los elementos escenciales de langchain, y ya podríamos avanzar hacia construir un chatbot. 

## 🤖 **Un chatbot básico**

A continuación comenzaremos a construir nuestro chatbot, incorporando progresivamente 2 elementos adicionales necesarios para un chatbot, que son contexto e historial de chat.

### 📖 **Contexto**

Los chatbots con LLM son una buena forma de proveer un servicio interactivo, ya sea con clientes o usuarios internos de una institución, que permita brindar información con instrucciones en lenguaje natural. 

Pero ¿Qué realmente podría aportar nuestro chatbot que ya no aportan otros servicios de LLM como Gemini, aparte de un rol e instrucciones específicas? En términos prácticos, uno también puede darle instrucciones a otro LLM para que te responda, y sería prácticamente idéntico a nuestro chatbot hecho con langchain.

Los chatbots institucionales o de empresas en general tienen como función entregar información y resolver dudas sobre esa institución, la cual debe estar actualizada. ¿Qué sucede si a un chatbot le preguntamos algo sobre actualidad?

In [60]:
response = llm.invoke("Quién gobierna Chile hoy?")
print(response.text)

El actual presidente de Chile es **Gabriel Boric Font**. Asumió el cargo el 11 de marzo de 2022.


**Pregunta**: ¿Porqué podría estar pasando esto? ¿Qué podemos hacer para arreglarlo?

In [61]:
response = llm.invoke("Cual es la fecha de hoy?")
print(response.text)

Hoy es miércoles 22 de mayo de 2024.


¿Cómo entonces los chatbots son capaces de responder sobre información no actualizada, la cual muchas veces no está abierta al público? La respuesta es: que utilizan contexto personalizado. Esto es, como parte de las instrucciones del bot se incorpora un contexto adicional que le entrega información que el usuario va a consultar.

A modo de ejemplo, utilizaremos el contexto presente en el [reglamento de estudiantes de la Universidad de Chile](./reglamento_u_de_chile.txt) para construir un chatbot que responda dudas sobre él. Cargaremos este archivo y lo incorporaremos completo en el prompt.

In [63]:
system_template = '''
Eres un asistente al estudiante. Tu rol es responder preguntas relacionadas con el reglamento de
estudiantes de la Universidad de Chile. Sólamente puedes responder preguntas sobre eso. Responde
con un tono que represente a nuestra prestigiosa universidad.

# Contexto

{contexto}
'''

with open("reglamento_u_de_chile.txt", encoding="utf-8") as f:
    reglamento_u_chile = f.read()


prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{question}")
])

# Pre llenamos con el contexto ya que no es un argumento del chatbot.
prompt_template = prompt_template.partial(contexto=reglamento_u_chile)

prompt_template


ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={'contexto': '# Reglamento de Estudiantes de la Universidad de Chile\n\nDecreto Universitario N°007586, de 19 de noviembre de 1993. Última modificación D.U. 0047282, 2016\n\nTITULO I\n\nDE LAS DISPOSICIONES GENERALES\n\nArtículo 1\n\nEl presente reglamento establece los deberes y derechos de los/las estudiantes, los aspectos relacionados con la participación estudiantil y la calidad de vida universitaria. Además, regula, en forma general los mecanismos de ingreso, permanencia y promoción de los/las estudiantes, salvo las excepciones que señale este propio reglamento.\n\nArtículo 2\n\nSon estudiantes de la Universidad quienes han formalizado su matrícula en carreras y programas académicos regulares y sistemáticos, de pre y posgrado, regulados en los respectivos reglamentos generales de estudios, y cumplen los requisitos establecidos por la Universidad para su ingreso, permanencia y promoción.\n\nSe recono

Luego, con el prompt_template podemos construir una chain que responda preguntas sobre el reglamento. 

> ⚠️**Cuidado:** Al usar una gran cantidad de contexto aumenta drásticamente el uso de tokens, y también aumentan los tiempos de respuesta considerablemente.

In [64]:
chain = prompt_template | llm | StrOutputParser()

result = chain.invoke("Puedo tomar en los pastos?")

print(result)

Estimado/a estudiante,

En relación con su consulta, le informo que el **Reglamento de Estudiantes de la Universidad de Chile** (Decreto Universitario N°007586) establece en su **Artículo 3, numeral 5**, que es deber de los/las estudiantes **cuidar el patrimonio y respetar los emblemas universitarios**.

Si bien el reglamento no menciona explícitamente el uso de los espacios verdes, nuestra normativa institucional promueve una convivencia universitaria basada en el respeto a la infraestructura y al entorno que nos permite desarrollar nuestra labor académica. Como miembros de esta prestigiosa casa de estudios, se espera que el comportamiento de cada estudiante contribuya a la preservación de nuestras instalaciones y al mantenimiento de un ambiente adecuado para la vida universitaria.

Le sugiero verificar si existen normativas específicas de su Facultad o Instituto respecto al uso de los espacios comunes, ya que, conforme al **Artículo 60** del presente reglamento, las situaciones no pr

En este caso, el reglamento del estudiante de la Universidad de Chile no es excesivamente grande por lo que puede inyectarse en el prompt sin mayores complicaciones. ¿Y qué puedo hacer si necesito que un chatbot responda sobre un contexto muy grande? Claramente existen límites para cuanto contexto puedo inyectar en un solo prompt, por lo que en este caso necesito alguna técnica más avanzada que lo permita. 

### 🔍 **RAG**

Una forma de evitar darle mucho contexto al chatbot sin tener que inyectar una cantidad gigantesca de texto en un solo prompt se llama **Retrieval Augmented Generation** o **RAG**. 

Pensemoslo en términos prácticos. La información que responde la pregunta del usuario probablemente solo se responde con una pequeña parte del texto de contexto. Sería ideal solo utilizar las partes que podrían contener la información deseada. Esto es básicamente RAG.

En esta técnica se divide el texto de contexto en múltiples "trozos", y estos se procesan con una técnica llamada 'embeddings' que permite representarlos semánticamente mediante vectores. Con esto, es sencillo hacer match entre la pregunta del usuario y los trozos de texto que contienen información similar. De esta forma, solo se utilizan los trozos de texto que podrían tener la información, optimizando el uso de tokens. Esta técnica la veremos en la próxima clase, ya que tratará sobre embeddings.

### 💬 **Historial de chat**

El otro elemento faltante es el historial de chat.

Actualmente hemos estado viendo chains que toman **el último mensaje** del usuario y lo utilizan para llamar a un LLM que le responda. Sin embargo, el LLM no tiene contexto de mensajes anteriores, por lo que podría pasar la siguiente situación:

- **Human**: ¿Cómo llego a puchuncaví?
- **AI**: ¡Hola! Para responderme cuéntame ¿Desde donde piensas viajar?
- **Human:** Desde Santiago
- **AI**: ¡Desde santiago puedes visitar muchos sitios interesantes! Cuéntame qué prefieres, ¿Montaña, playa o campo?

Para que el bot realmente pueda responder la pregunta debe saber el contexto de la conversación, así podría internet qué hacer con un mensaje como "Sí". Ese mensaje en sí casi no contiene información. Pero el conjunto de mensajes "Hola, dame el tiempo - El tiempo de Hoy? - Sí" Contiene muchísima información. Para que esto suceda, el chatbot debe de alguna forma almacenar los mensajes para tener claridad del contexto.

En primer lugar, probaremos la funcionalidad de generar un prompt con chat history. Utilizaremos HumanMessage (rol=human) y AIMessage (rol=model) para simular mensajes anteriores del usuario y el chatbot (no son prompt template).

In [65]:
from langchain_core.messages import HumanMessage, AIMessage

system_template = '''
Eres un asistente al estudiante. Tu rol es responder preguntas relacionadas con el reglamento de
estudiantes de la Universidad de Chile. Sólamente puedes responder preguntas sobre eso. Responde
con un tono que represente a nuestra prestigiosa universidad.

# Contexto

{contexto}
'''

with open("reglamento_u_de_chile.txt", encoding="utf-8") as f:
    reglamento_u_chile = f.read()

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    HumanMessage("Hola, se puede mechonear?"),
    AIMessage("Estimado/a estudiante. No está permitido. Saludos"),
    HumanMessage("{question}"),
])

prompt_template = prompt_template.partial(contexto=reglamento_u_chile)
chain = prompt_template | llm | StrOutputParser()

In [66]:
result = chain.invoke({"question": "Por qué no?"})

print(result)

Estimado/a estudiante,

En relación con su consulta, es imperativo señalar que el **Reglamento de Estudiantes de la Universidad de Chile** establece en su **Artículo 3, numeral 4**, el deber de respetar a todos los integrantes de la comunidad universitaria.

Asimismo, el **Artículo 5** del mismo cuerpo normativo dispone que constituye infracción todo comportamiento que importe la transgresión a los deberes establecidos en el artículo 3, lo cual incluye cualquier acto que atente contra la integridad o dignidad de los miembros de nuestra comunidad.

Por lo tanto, cualquier práctica que vulnere el respeto y la convivencia universitaria, como la que usted menciona, es contraria a la normativa institucional y se encuentra sujeta a las sanciones que determine el Reglamento de Jurisdicción Disciplinaria de los Estudiantes.

Le instamos a contribuir a una convivencia universitaria basada en el respeto mutuo, pilar fundamental de nuestra prestigiosa institución.


Vemos que a pesar de que la pregunta es 'Por qué no?' el chatbot es capaz de responder ya que tiene como contexto preguntas anteriores que apuntaban a poder _tomar en los pastos_. 

Para que funcione realmente como un chatbot, este debe ser capaz de almacenar dinámicamente los mensajes tanto de humano como de AI en un historial de chat. Realizaremos esto de forma rudimentaria en python pero compeltamente funciona, construyendo nuestro primer chatbot funcional!

In [68]:
class ChatbotSoporteEstudiantesUChile:
    def __init__(self, prompt, contexto, model_name="gemini-3.1-flash-lite", nombre="Uchi"):
        self.nombre = nombre
        self.model_name = model_name
        self.prompt = prompt
        self.contexto = contexto
        self.chat_history = []
        self.prompt_template = SystemMessagePromptTemplate.from_template(self.prompt)

        self.llm = ChatGoogleGenerativeAI(
            model=model_name, 
            max_retries=2,
        )

    def add_message(self, message, user=True):
        if user:
            new_message = HumanMessage(content=message)
        else:
            new_message = AIMessage(content=message)

        self.chat_history.append(new_message)

    def setup_prompt_template(self):
        messages = [self.prompt_template] + self.chat_history
        messages.append(HumanMessagePromptTemplate.from_template("{question}"))
        prompt_template = ChatPromptTemplate.from_messages(messages)
        prompt_template = prompt_template.partial(contexto=self.contexto)

        return prompt_template
    
    def chat(self, mensaje_usuario):
        self.add_message(mensaje_usuario)
        prompt_template = self.setup_prompt_template()

        chain = prompt_template | self.llm | StrOutputParser()
        response = chain.invoke({"question": mensaje_usuario})

        self.add_message(response, user=False)

        return response
        

Instanciamos nuestro chatbot una primera vez para que genere memoria y luego la reutilice

In [69]:
chatbot = ChatbotSoporteEstudiantesUChile(prompt=system_template, contexto=reglamento_u_chile)

### **A chatear!**

In [71]:
pregunta = "Dime mas del punto 2"

respuesta = chatbot.chat(pregunta)

print(respuesta)

Estimado/a estudiante, es un honor profundizar en el **numeral 8 del Artículo 4** de nuestro Reglamento, el cual consagra el derecho a una **adecuada calidad de vida estudiantil**. Para nuestra Universidad, el bienestar del estudiantado es un pilar indispensable para alcanzar la excelencia académica.

Este derecho se estructura en cuatro dimensiones fundamentales que le permiten desarrollarse de manera integral:

*   **Desarrollo integral (Literal a):** La Universidad se compromete a facilitar la práctica de actividades deportivas, culturales, recreativas y de desarrollo personal, siempre en concordancia con sus condiciones curriculares. Entendemos que su formación trasciende el aula.
*   **Bienestar y Salud (Literal b):** Usted tiene el derecho de recibir atención en salud y asistencia social, conforme a las condiciones y prestaciones que nuestra institución provee para el estamento estudiantil.
*   **Ayudas Estudiantiles (Literal c):** El acceso a la educación pública es un compromis

<center>
<img src='https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExYncycHI0NDJlZG1tdndpY3ZtcjhweTgzZmwyNjliNHVyNnMwZTE4diZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/FRWvWoLoL9311DTflT/giphy.gif' width=450  />
</center>
